# DATA WRANGLING

## SETUP

In [ ]:
import duckdb
import pandas as pd
import sqlite3
from pathlib import Path
import holidays


try:
    # if running as a .py, __file__ exists
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    # if running in a notebook, __file__ does NOT exist
    BASE_DIR = Path.cwd().parents[1] 

# path to database
DATA_DIR = BASE_DIR / "data" / "train.duckdb"
print(DATA_DIR)

/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project/data/train.duckdb


## IMPORT DATA

In [3]:
# establish SQL connection to database and load into dataframe
with duckdb.connect(DATA_DIR) as con:
    df_raw = con.execute("SELECT * FROM train_delay").df()

## DATA INSPECTION

In [4]:
print(df_raw.info())                      
# print(df_raw.head(5))                      
# print(df_raw["delay_in_min"].describe())  
# print(df_raw["train_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3567509 entries, 0 to 3567508
Data columns (total 14 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   eva                        object        
 1   station_name               object        
 2   final_destination_station  object        
 3   train_name                 object        
 4   train_type                 object        
 5   departure_planned_time     datetime64[ns]
 6   departure_change_time      datetime64[ns]
 7   arrival_planned_time       datetime64[ns]
 8   arrival_change_time        datetime64[ns]
 9   delay_in_min               int32         
 10  is_canceled                bool          
 11  prev_time                  datetime64[ns]
 12  journey_id_train           float64       
 13  journey_id                 int64         
dtypes: bool(1), datetime64[ns](5), float64(1), int32(1), int64(1), object(5)
memory usage: 343.6+ MB
None


## REMOVE AND RENAME COLUMNS

In [6]:
# remove unnecessary columns
columns_to_drop = [
    "eva",
    "arrival_change_time",
    "prev_time",
    "journey_id_train"]
df = df_raw.drop(columns=columns_to_drop)

# rename columns
df = df.rename(columns={
    "station_name": "station_current",
    "final_destination_station": "station_dest",
    "departure_planned_time": "time_current_departure_planned",
    "departure_change_time": "time_current_departure_real",
    "arrival_planned_time": "time_current_arrival_planned",
    "delay_in_min": "delay",
    "is_canceled": "canceled"})


# rearrange columns
df = df.loc[:, ["id", "journey_id", "train_name", "train_type", "station_current", "station_dest",
               "time_current_departure_planned", "time_current_departure_real", "time_current_arrival_planned", "delay", "canceled"]]

print(df.columns.tolist())
print(df.info())    


['id', 'journey_id', 'train_name', 'train_type', 'station_current', 'station_dest', 'time_current_departure_planned', 'time_current_departure_real', 'time_current_arrival_planned', 'delay', 'canceled']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3551715 entries, 0 to 3551714
Data columns (total 11 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   id                              int64         
 1   journey_id                      int64         
 2   train_name                      object        
 3   train_type                      object        
 4   station_current                 object        
 5   station_dest                    object        
 6   time_current_departure_planned  datetime64[ns]
 7   time_current_departure_real     datetime64[ns]
 8   time_current_arrival_planned    datetime64[ns]
 9   delay                           int32         
 10  canceled                        bool          
dtypes: b

## DATA-TYPES

In [7]:
# categorical variables
categorical_cols = [
    "journey_id",
    "train_name",
    "train_type",
    "station_current",
    "station_dest"]
df[categorical_cols] = df[categorical_cols].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3551715 entries, 0 to 3551714
Data columns (total 11 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   id                              int64         
 1   journey_id                      category      
 2   train_name                      category      
 3   train_type                      category      
 4   station_current                 category      
 5   station_dest                    category      
 6   time_current_departure_planned  datetime64[ns]
 7   time_current_departure_real     datetime64[ns]
 8   time_current_arrival_planned    datetime64[ns]
 9   delay                           int32         
 10  canceled                        bool          
dtypes: bool(1), category(5), datetime64[ns](3), int32(1), int64(1)
memory usage: 185.0 MB


## RIDE-RELATED

### STATION NAMES, TIME AND STOPS

In [8]:
features_ride = (
    df
    .groupby("journey_id") 
    .agg(     
        # route
        station_start=("station_current", "first"),
        stops_total=("station_current", "count"),
        
        # time
        time_start_planned=("time_current_departure_planned", "first"),
        time_dest_planned=("time_current_arrival_planned", "last"),
    )
    .reset_index())

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/4211776477.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("journey_id")


### TRAVEL TIME

In [9]:
features_ride["travel_time"] = (
    features_ride["time_dest_planned"]
    - features_ride["time_start_planned"]
).dt.total_seconds() / 60

### CALENDAR_RELATED


In [10]:
features_ride = features_ride.assign(
    weekday = lambda d: d["time_start_planned"].dt.weekday.astype("category"),  
    weekend = lambda d: (d["weekday"].isin([5, 6])).astype("category"),
    month = lambda d: d["time_start_planned"].dt.month.astype("category")
)

### SEASON

In [11]:
# function that assigns season to month
def season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"
    
features_ride["season"] = features_ride["month"].apply(season).astype("category")

### FEAST

In [12]:
# get holidays
de_holidays = holidays.Germany()

'''
features_ride["feast"] = (
    features_ride["time_start_planned"]
    .dt.date
    .apply(lambda d: int(d in de_holidays))
    .astype(int)
)
'''

features_ride["feast"] = (
    features_ride["time_start_planned"]
    .dt.date
    .apply(lambda d: 1 if pd.notnull(d) and d in de_holidays else 0)
)


features_ride.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 805479 entries, 0 to 805478
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   journey_id          805479 non-null  category      
 1   station_start       805479 non-null  category      
 2   stops_total         805479 non-null  int64         
 3   time_start_planned  444114 non-null  datetime64[ns]
 4   time_dest_planned   786809 non-null  datetime64[ns]
 5   travel_time         425444 non-null  float64       
 6   weekday             444114 non-null  category      
 7   weekend             805479 non-null  category      
 8   month               444114 non-null  category      
 9   season              444114 non-null  category      
 10  feast               805479 non-null  int64         
dtypes: category(6), datetime64[ns](2), float64(1), int64(2)
memory usage: 60.7 MB


## STATION-RELATED

### DWELL TIME

In [13]:
# dwell time is difference between planned departure and arrival time at current station
features_station = df.copy()
features_station["dwell_time_planned"] = (
    features_station["time_current_departure_planned"]
    - features_station["time_current_arrival_planned"]
).dt.total_seconds() / 60    

### HISTORICAL AGGREGATION

In [15]:
# function to add expanding features
def add_expanding_features(df, group_cols, prefix):
    df = df.sort_values(group_cols + ["time_current_departure_real"]) # sort by real time because otherwise expanding stats might include info from future

    grouped = df.groupby(group_cols, sort=False)["delay"]

    df[f"{prefix}_avg"] = grouped.transform(lambda x: x.shift().expanding().mean())
    df[f"{prefix}_std"] = grouped.transform(lambda x: x.shift().expanding().std())
    df[f"{prefix}_q90"] = grouped.transform(lambda x: x.shift().expanding().quantile(0.9))
    df[f"{prefix}_count"] = grouped.transform(lambda x: x.shift().expanding().count()
)

    return df

# derive time features from real time
features_station["hour"] = features_station["time_current_departure_real"].dt.hour
features_station["weekday"] = features_station["time_current_departure_real"].dt.dayofweek

# apply expanding features
features_station = add_expanding_features(features_station, ["station_current"], "hist_delay_station")   
features_station = add_expanding_features(features_station, ["station_current", "hour"], "hist_delay_station_hour")  
features_station = add_expanding_features(features_station, ["station_current", "weekday"], "hist_delay_station_day")  

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(group_cols, sort=False)["delay"]
/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby(group_cols, sort=False)["delay"]
/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/1970335331.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or ob

### DROP UNNECESSARY VARIABLES

In [16]:
features_station.columns.tolist()

['id',
 'journey_id',
 'train_name',
 'train_type',
 'station_current',
 'station_dest',
 'time_current_departure_planned',
 'time_current_departure_real',
 'time_current_arrival_planned',
 'delay',
 'canceled',
 'dwell_time_planned',
 'hour',
 'weekday',
 'hist_delay_station_avg',
 'hist_delay_station_std',
 'hist_delay_station_q90',
 'hist_delay_station_count',
 'hist_delay_station_hour_avg',
 'hist_delay_station_hour_std',
 'hist_delay_station_hour_q90',
 'hist_delay_station_hour_count',
 'hist_delay_station_day_avg',
 'hist_delay_station_day_std',
 'hist_delay_station_day_q90',
 'hist_delay_station_day_count']

In [18]:
columns_to_drop = [
    "journey_id",
    "train_name",
    "train_type",
    "station_current",
    "station_dest",
    "time_current_arrival_planned",
    "time_current_departure_planned",
    "time_current_departure_real",
    "delay",
    "canceled",
    "weekday"]
features_station.drop(columns=columns_to_drop, inplace=True)

## SEQUENCE-RELATED

### JOIN INFORMATION

In [23]:
# join features back to main dataframe
df_full = df.merge(
    features_ride,
    on = "journey_id",
    how = "left"
)
df_full = df_full.merge(
    features_station,
    on = "id",
    how = "left"
)
print(df_full.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3551715 entries, 0 to 3551714
Data columns (total 35 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   id                              int64         
 1   journey_id                      category      
 2   train_name                      category      
 3   train_type                      category      
 4   station_current                 category      
 5   station_dest                    category      
 6   time_current_departure_planned  datetime64[ns]
 7   time_current_departure_real     datetime64[ns]
 8   time_current_arrival_planned    datetime64[ns]
 9   delay                           int32         
 10  canceled                        bool          
 11  station_start                   category      
 12  stops_total                     int64         
 13  time_start_planned              datetime64[ns]
 14  time_dest_planned               datetime64[ns]
 15

### SEQUENCE VARS

In [24]:
# station role: start, mid, end
df_full["station_role"] = df_full.apply(
    lambda row: "start" if row["station_current"] == row["station_start"] else (
        "end" if row["station_current"] == row["station_dest"] else "mid"), axis=1).astype("category")    

# stops index in journey
df_full["stop_index"] = df_full.groupby("journey_id").cumcount() + 1    

# stops remaining to destination
df_full["stops_remaining"] = df_full["stops_total"] - df_full["stop_index"]

# time since start planned
df_full["time_since_start_planned"] = (
    df_full["time_current_arrival_planned"]
    - df_full["time_start_planned"]
).dt.total_seconds() / 60   

# time to destination planned
df_full["time_to_dest_planned"] = (
    df_full["time_dest_planned"]
    - df_full["time_current_departure_planned"]
).dt.total_seconds() / 60

# share ride time
df_full["share_ride_time_completed"] = (
    df_full["time_since_start_planned"] / df_full["travel_time"]
)

# share ride stations
df_full["share_ride_stations"] = (
    df_full["stop_index"] / df_full["stops_total"]
)

/var/folders/9n/ws1j083x5m9d1jv6n898lzlw0000gn/T/ipykernel_14285/2729890663.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_full["stop_index"] = df_full.groupby("journey_id").cumcount() + 1


## SAVE DATASET

In [ ]:
# save final dataframe
OUTPUT_DIR = BASE_DIR / "data" / "train_delay_processed.duckdb"
with duckdb.connect(OUTPUT_DIR) as con:
    con.execute("DROP TABLE IF EXISTS train_delay_processed")
    con.execute("CREATE TABLE train_delay_processed AS SELECT * FROM df_final") 